# LLM Provider Complete Guide

ADLab supports multiple LLM Providers including OpenAI, Claude, vLLM, and SGLang.

## Table of Contents
1. [LanguageModel Base Class](#1-LanguageModel-Base-Class)
2. [OpenAI API](#2-OpenAI-API)
3. [Claude API](#3-Claude-API)
4. [vLLM and SGLang](#4-vLLM-and-SGLang)
5. [Usage Examples](#5-Usage-Examples)
6. [Error Handling](#6-Error-Handling)
7. [Rate Limiting](#7-Rate-Limiting)
8. [Async Usage](#8-Async-Usage)
9. [YAML Configuration](#9-YAML-Configuration)

## 1. LanguageModel Base Class

The `LanguageModel` abstract base class defines the interface that all LLM providers must implement. This ensures a consistent API across different backends.

In [ ]:
from autodiscover.base.llm import LanguageModel
import inspect

# View LanguageModel methods
print("LanguageModel Methods:")
for name, method in inspect.getmembers(LanguageModel, predicate=inspect.isfunction):
    if not name.startswith('_'):
        print(f"  - {name}")

## 2. OpenAI API

The OpenAI provider supports GPT models via the official OpenAI API or compatible endpoints (e.g., proxies, Azure OpenAI).

### Initialization Options
- **Environment Variables**: Set `OPENAI_API_KEY` and `OPENAI_BASE_URL`
- **Direct Parameters**: Pass `api_key` and `base_url` directly to the constructor

### Supported Models
- `gpt-3.5-turbo` - Fast, cost-effective for most tasks
- `gpt-4` - More capable, slower and more expensive
- `gpt-4-turbo` - Optimized for speed and cost
- Any OpenAI-compatible model endpoint

In [ ]:
import os

# Set API Key (replace with your actual key)
os.environ["OPENAI_API_KEY"] = "your-api-key"
os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"

from autodiscover.providers.llm import OpenAIAPI

# Initialize - api_key and base_url can be omitted if set via environment variables
llm = OpenAIAPI(
    model="gpt-3.5-turbo",
    # base_url and api_key are optional if using environment variables
)

## 3. Claude API

The Claude provider uses Anthropic's Claude models through the official API.

### Initialization Options
- **Environment Variable**: Set `ANTHROPIC_API_KEY`
- **Direct Parameter**: Pass `api_key` to the constructor

### Supported Models
- `claude-3-opus-20240229` - Most capable, highest cost
- `claude-3-sonnet-20240229` - Balanced performance and cost
- `claude-3-haiku-20240229` - Fastest, lowest cost
- `claude-3-5-sonnet-20241022` - Latest Sonnet model

In [ ]:
from autodiscover.providers.llm import ClaudeAPI

# Initialize Claude
claude = ClaudeAPI(
    model="claude-3-sonnet-20240229",
    api_key="your-anthropic-api-key"  # Or set ANTHROPIC_API_KEY env var
)

# Note: Claude also supports setting via ANTHROPIC_API_KEY environment variable

## 4. vLLM and SGLang

vLLM and SGLang are locally deployed LLM servers that provide OpenAI-compatible APIs. They are ideal for:
- High-volume inference at lower cost
- Privacy-sensitive workloads (data stays local)
- Custom model deployment

### Starting a vLLM Server
```bash
vllm serve Q/Qwen2.5-0.5B-Instruct --dtype half --api-key token-abc123
```

### Starting a SGLang Server
```bash
python -m sglang.launch_server --model Q/Qwen2.5-0.5B-Instruct --dtype half --api-key token-abc123
```

In [ ]:
from autodiscover.providers.llm import VLLMServer, SGLangServer

# vLLM initialization
vllm = VLLMServer(
    model="Q/Qwen2.5-0.5B-Instruct",
    base_url="http://localhost:8000/v1",
    api_key="token-abc123"
)

# SGLang initialization
sglang = SGLangServer(
    model="Q/Qwen2.5-0.5B-Instruct",
    base_url="http://localhost:30000/v1",
    api_key="token-abc123"
)

## 5. Usage Examples

### 5.1 Basic Chat Completion

The unified `chat_completion` interface works across all providers.

In [ ]:
# Unified interface - chat_completion

# Basic usage - string message
response = llm.chat_completion(
    message="Hello, how are you?",
    max_tokens=100,
    timeout_seconds=30
)

print(f"Response: {response[:100]}...")

In [ ]:
# Advanced usage - message list (multi-turn conversation)

messages = [
    {"role": "system", "content": "You are a helpful coding assistant."},
    {"role": "user", "content": "Write a Python function to sort a list."},
]

response = llm.chat_completion(
    message=messages,
    max_tokens=200,
    timeout_seconds=30
)

print(f"Response: {response[:200]}...")

In [ ]:
# Using additional parameters

response = llm.chat_completion(
    message="Write a haiku about programming",
    max_tokens=50,
    temperature=0.7,  # Controls randomness (0.0-2.0, higher = more creative)
    top_p=0.9,       # Nucleus sampling threshold
)

print(f"Response: {response}")

In [ ]:
# Embedding usage

# Single text
embedding = llm.embedding(
    text="Hello world",
    dimensions=1536
)
print(f"Embedding dimensions: {len(embedding)}")

# Multiple texts (batch processing)
embeddings = llm.embedding(
    text=["Hello", "World", "Test"]
)
print(f"Number of embeddings: {len(embeddings)}")

In [ ]:
# Resource management - close()

# Always close providers when done to free resources
llm.close()
claude.close()
vllm.close()
sglang.close()

# Or use context manager (if __enter__/__exit__ is implemented)
# with OpenAIAPI(model="gpt-3.5-turbo") as llm:
#     response = llm.chat_completion("Hello")

## 6. Error Handling

API calls can fail due to various reasons. Here's how to handle common errors gracefully.

In [ ]:
import time
import json
from autodiscover.providers.llm import OpenAIAPI

def retry_with_backoff(
    func,
    max_retries=3,
    initial_backoff=1.0,
    max_backoff=60.0,
    backoff_factor=2.0
):
    """
    Retry a function with exponential backoff on failure.
    
    Args:
        func: Function to retry
        max_retries: Maximum number of retry attempts
        initial_backoff: Initial backoff time in seconds
        max_backoff: Maximum backoff time in seconds
        backoff_factor: Multiplier for each retry
    """
    for attempt in range(max_retries):
        try:
            return func()
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            
            backoff = min(initial_backoff * (backoff_factor ** attempt), max_backoff)
            print(f"Attempt {attempt + 1} failed: {e}. Retrying in {backoff}s...")
            time.sleep(backoff)


# Example: Handling API failures
llm = OpenAIAPI(model="gpt-3.5-turbo")

def make_api_call():
    try:
        response = llm.chat_completion(
            message="Explain quantum computing in simple terms.",
            max_tokens=200,
            timeout_seconds=30
        )
        return response
    except Exception as e:
        # Log the error for debugging
        print(f"API call failed: {type(e).__name__}: {e}")
        raise

try:
    result = retry_with_backoff(make_api_call, max_retries=3)
    print(f"Success: {result[:100]}...")
except Exception as e:
    print(f"All retries exhausted. Final error: {e}")
finally:
    llm.close()

In [ ]:
# Handling specific error types

class LLMError(Exception):
    """Base exception for LLM-related errors."""
    pass

class RateLimitError(LLMError):
    """Raised when API rate limit is exceeded."""
    pass

class AuthenticationError(LLMError):
    """Raised when API key is invalid or missing."""
    pass

class APITimeoutError(LLMError):
    """Raised when API request times out."""
    pass


def safe_chat_completion(llm, message, max_retries=3):
    """
    Safely call chat_completion with error handling.
    
    Returns:
        tuple: (success: bool, response: str or None, error: str or None)
    """
    try:
        response = llm.chat_completion(
            message=message,
            max_tokens=100,
            timeout_seconds=30
        )
        return True, response, None
    except Exception as e:
        error_type = type(e).__name__
        
        # Categorize common errors
        if "rate limit" in str(e).lower() or "429" in str(e):
            return False, None, f"RateLimitError: {e}"
        elif "auth" in str(e).lower() or "401" in str(e):
            return False, None, f"AuthenticationError: {e}"
        elif "timeout" in str(e).lower() or "timed out" in str(e).lower():
            return False, None, f"APITimeoutError: {e}"
        else:
            return False, None, f"{error_type}: {e}"

## 7. Rate Limiting

Different providers have different rate limits. Here's how to manage them.

In [ ]:
import time
from collections import deque
from threading import Lock


class RateLimiter:
    """
    Token bucket rate limiter for API calls.
    
    Ensures requests stay within rate limits using the token bucket algorithm.
    """
    
    def __init__(self, requests_per_minute: int = 60, requests_per_day: int = None):
        """
        Initialize rate limiter.
        
        Args:
            requests_per_minute: Maximum requests allowed per minute
            requests_per_day: Optional daily limit override
        """
        self.requests_per_minute = requests_per_minute
        self.requests_per_day = requests_per_day
        self.minute_buckets = deque(maxlen=60)  # Track recent requests
        self.daily_requests = [] if requests_per_day else None
        self.lock = Lock()
    
    def acquire(self, timeout=None):
        """
        Acquire permission to make a request.
        
        Blocks until request is allowed or timeout is reached.
        
        Args:
            timeout: Maximum time to wait in seconds (None = wait forever)
            
        Returns:
            bool: True if permit acquired, False if timeout
        """
        start_time = time.time()
        
        while True:
            with self.lock:
                now = time.time()
                
                # Clean up old minute entries
                while self.minute_buckets and now - self.minute_buckets[0] > 60:
                    self.minute_buckets.popleft()
                
                # Check daily limit
                if self.requests_per_day:
                    day_start = now - 86400  # 24 hours ago
                    self.daily_requests = [t for t in self.daily_requests if t > day_start]
                    if len(self.daily_requests) >= self.requests_per_day:
                        wait_time = self.daily_requests[0] + 86400 - now
                        if timeout and (time.time() - start_time + wait_time) > timeout:
                            return False
                        time.sleep(min(wait_time, 1))
                        continue
                
                # Check minute limit
                if len(self.minute_buckets) < self.requests_per_minute:
                    self.minute_buckets.append(now)
                    if self.requests_per_day:
                        self.daily_requests.append(now)
                    return True
            
            # Wait before checking again
            wait_time = 60 - (now - self.minute_buckets[0]) if self.minute_buckets else 1
            if timeout and (time.time() - start_time + wait_time) > timeout:
                return False
            time.sleep(min(wait_time, 1))


# Example usage with rate limiting
rate_limiter = RateLimiter(requests_per_minute=30)  # 30 requests/minute

def rate_limited_completion(llm, message):
    """Make a rate-limited API call."""
    if rate_limiter.acquire(timeout=60):
        try:
            return llm.chat_completion(message=message, max_tokens=100)
        except Exception as e:
            print(f"Request failed: {e}")
            raise
    else:
        raise TimeoutError("Rate limiter timeout - could not acquire permit in time")

# Simulated batch processing with rate limiting
messages = [f"Request {i}" for i in range(5)]

for msg in messages:
    try:
        result = rate_limited_completion(llm, msg)
        print(f"Success: {result[:50]}...")
    except Exception as e:
        print(f"Failed: {e}")

## 8. Async Usage

For high-throughput applications, async patterns allow concurrent API calls.

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
from autodiscover.providers.llm import OpenAIAPI


def async_chat_completion(llm, message, max_tokens=100, timeout_seconds=30):
    """
    Wrapper to use chat_completion in an async context.
    
    This runs the blocking API call in a thread pool to avoid blocking the event loop.
    """
    return llm.chat_completion(
        message=message,
        max_tokens=max_tokens,
        timeout_seconds=timeout_seconds
    )

async def batch_chat_completion_async(messages, max_concurrent=5):
    """
    Process multiple chat completions concurrently with a semaphore limit.
    
    Args:
        messages: List of message strings to process
        max_concurrent: Maximum number of concurrent API calls
    """
    llm = OpenAIAPI(model="gpt-3.5-turbo")
    
    async def process_with_semaphore(message, semaphore):
        async with semaphore:
            loop = asyncio.get_event_loop()
            result = await loop.run_in_executor(
                None,
                lambda: async_chat_completion(llm, message)
            )
            return result
    
    semaphore = asyncio.Semaphore(max_concurrent)
    
    try:
        tasks = [process_with_semaphore(msg, semaphore) for msg in messages]
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        # Process results
        successful = [r for r in results if not isinstance(r, Exception)]
        failed = [(i, r) for i, r in enumerate(results) if isinstance(r, Exception)]
        
        print(f"Completed: {len(successful)} successful, {len(failed)} failed")
        
        for i, error in failed:
            print(f"  Message {i} failed: {error}")
        
        return results
        
    finally:
        llm.close()


# Run async batch processing
messages = [
    "What is Python?",
    "What is an API?",
    "Explain machine learning",
    "What is a neural network?",
    "Define deep learning"
]

# Note: In a Jupyter notebook, use asyncio.run()
# In a Python script, you can also use await directly
results = asyncio.run(batch_chat_completion_async(messages, max_concurrent=3))
print(f"\nProcessed {len(results)} messages")

In [ ]:
# ThreadPoolExecutor for parallel sync calls

from concurrent.futures import ThreadPoolExecutor, as_completed


def parallel_chat_completions(llm, messages, max_workers=5):
    """
    Process multiple chat completions in parallel using ThreadPoolExecutor.
    
    Args:
        llm: LLM provider instance
        messages: List of message strings
        max_workers: Maximum number of concurrent threads
    
    Returns:
        list: List of (message_index, response_or_error) tuples
    """
    results = []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {
            executor.submit(llm.chat_completion, msg, 100, 30): i
            for i, msg in enumerate(messages)
        }
        
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                response = future.result()
                results.append((idx, response))
            except Exception as e:
                results.append((idx, f"Error: {e}"))
    
    # Sort by original index
    results.sort(key=lambda x: x[0])
    return results


# Example: parallel processing
llm = OpenAIAPI(model="gpt-3.5-turbo")
messages = [
    "Hello!",
    "How are you?",
    "What is 2+2?",
]

results = parallel_chat_completions(llm, messages, max_workers=3)

for idx, response in results:
    print(f"Message {idx}: {response[:50]}..." if len(str(response)) > 50 else f"Message {idx}: {response}")

llm.close()

## 9. YAML Configuration

LLM providers can be configured via YAML for use in config files. This is the recommended approach for production deployments.

In [ ]:
# OpenAI Configuration
openai_config = """
llm:
  class_path: "autodiscover.providers.llm.openai_api.OpenAIAPI"
  kwargs:
    model: "gpt-3.5-turbo"
    api_key: "your-api-key"  # Or set via OPENAI_API_KEY env var
    base_url: "https://api.openai.com/v1"
"""

# Claude Configuration
claude_config = """
llm:
  class_path: "autodiscover.providers.llm.claude_api.ClaudeAPI"
  kwargs:
    model: "claude-3-sonnet-20240229"
    api_key: "your-anthropic-api-key"  # Or set via ANTHROPIC_API_KEY env var
"""

# vLLM Configuration (local deployment)
vllm_config = """
llm:
  class_path: "autodiscover.providers.llm.vllm_server.VLLMServer"
  kwargs:
    model: "Q/Qwen2.5-0.5B-Instruct"
    base_url: "http://localhost:8000/v1"
    api_key: "token-abc123"
"""

# SGLang Configuration (local deployment)
sglang_config = """
llm:
  class_path: "autodiscover.providers.llm.sglang_server.SGLangServer"
  kwargs:
    model: "Q/Qwen2.5-0.5B-Instruct"
    base_url: "http://localhost:30000/v1"
    api_key: "token-abc123"
"""

print("YAML Configuration Examples:")
print(openai_config)

## Summary

### Provider Comparison

| Provider | Class | Use Case |
|----------|-------|----------|
| OpenAI | `OpenAIAPI` | General purpose, OpenAI API compatible endpoints |
| Claude | `ClaudeAPI` | Anthropic Claude models, high-quality reasoning |
| vLLM | `VLLMServer` | Local vLLM servers, high-volume low-cost inference |
| SGLang | `SGLangServer` | Local SGLang servers, optimized inference |

### Common Methods

| Method | Description |
|--------|-------------|
| `chat_completion()` | Send a chat completion request |
| `embedding()` | Generate text embeddings |
| `close()` | Release resources |

### Best Practices

1. **Use environment variables** for API keys to avoid hardcoding
2. **Implement retry logic** with exponential backoff for resilience
3. **Use rate limiting** to stay within API quotas
4. **Close providers** when done to free connections
5. **Use async patterns** for high-throughput scenarios